In [13]:
from pprint import pprint
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.redis import RedisSaver
from langgraph.store.redis import RedisStore

from agentic_patterns.react_agent.agent import REACT_AGENT_BUILDER
redis_url = "redis://localhost:6379/0"
thread_id = "019db05b-da01-72e0-bf19-b0ebe0e3d3a3"
with RedisStore.from_conn_string(redis_url) as store:
    with RedisSaver.from_conn_string(redis_url) as ch:
        config = RunnableConfig(configurable={"thread_id": thread_id})        
        agent = REACT_AGENT_BUILDER.compile(checkpointer=ch, store=store)

        history = [*agent.get_state_history(config=config)]
        # pprint(history, indent=2)
        if not history:
            print(f"No history found for thread_id {thread_id}.")
        else:
            last_state = history[0]
            print(len(last_state.values.get("messages") or []))
            pprint(last_state.interrupts, indent=2)
        # ch.delete_thread(thread_id)
         

17
( Interrupt(value=[ { 'options': [ 'Yes, marts are in place',
                                   'Partially built',
                                   'Starting from scratch'],
                      'question': 'Do you already have mart models in place, '
                                  'or are you starting from scratch?'}],
            id='e37b8aa20be0d6c177e72f0f3c3d95d1'),)


In [9]:
from pprint import pprint

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.redis import AsyncRedisSaver
from langgraph.store.redis import AsyncRedisStore
from langgraph.graph.state import CompiledStateGraph
from redis.asyncio import Redis

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import (
    REDIS_URL,
    build_expert_tools,
    install_safe_pending_sends_loader,
    make_parent,
    make_subagents,
)

MODE = "own_same_thread"
parent_thread_id = f"ckpt-mode-{MODE}"
parent_config = RunnableConfig(configurable={"thread_id": parent_thread_id})

install_safe_pending_sends_loader()

# 1. Discover every (thread_id, checkpoint_ns) location that contains a
#    checkpoint related to this parent thread — covers both same-thread+sub-ns
#    and new-thread (parent-fruit / parent-veggie) layouts.
async with Redis.from_url(REDIS_URL, decode_responses=True) as r:
    ck_keys = sorted([k async for k in r.scan_iter(match=f"checkpoint:*{parent_thread_id}*")])

locations: set[tuple[str, str]] = set()
for k in ck_keys:
    parts = k.split(":")
    # checkpoint : <thread> : <ns_part_1> : ... : <ckpt_id>
    thread, ns = parts[1], ":".join(parts[2:-1])
    locations.add((thread, ns))

print(f"discovered {len(locations)} (thread_id, checkpoint_ns) location(s):")
for t, n in sorted(locations):
    print(f"  thread={t}  ns={n!r}")

# 2. Parent history through the compiled graph (gives StateSnapshots with `next`/`tasks`).
async with (
    AsyncRedisStore.from_conn_string(REDIS_URL) as store,
    AsyncRedisSaver.from_conn_string(REDIS_URL) as saver,
    AsyncRedisSaver.from_conn_string(REDIS_URL) as sub_saver,
):
    await store.setup()
    await saver.asetup()
    await sub_saver.asetup()

    # subagents are still needed for the parent to compile (the @tool wrapper
    # closes over them), but we will NOT call .aget_state_history on them.
    fruit_agent, veggie_agent = make_subagents(MODE, sub_saver)
    expert_tools = build_expert_tools(fruit_agent, veggie_agent, MODE)
    parent: CompiledStateGraph = make_parent(saver, store, expert_tools)

    parent_history = [s async for s in parent.aget_state_history(parent_config)]

    # 3. Sub state via the saver directly — no compiled subagent needed.
    sub_tuples_by_loc: dict[tuple[str, str], list] = {}
    for thread, ns in sorted(locations):
        if (thread, ns) == (parent_thread_id, ""):
            continue  # that's the parent, already covered
        cfg = RunnableConfig(configurable={"thread_id": thread, "checkpoint_ns": ns})
        sub_tuples_by_loc[(thread, ns)] = [t async for t in saver.alist(cfg)]

print(f"\nparent history length: {len(parent_history)}")
for i, snap in enumerate(reversed(parent_history)):
    msgs = snap.values.get("messages") or []
    print(
        f"  [{i}] step={snap.metadata.get('step'):<3} "
        f"source={snap.metadata.get('source'):<10} "
        f"next={snap.next}  msgs={len(msgs)}"
    )

print("\nsub state (read via saver, no compiled subagent):")
for (thread, ns), tuples in sub_tuples_by_loc.items():
    print(f"  thread={thread}  ns={ns!r}  tuples={len(tuples)}")
    for j, tup in enumerate(reversed(tuples)):
        msgs = tup.checkpoint.get("channel_values", {}).get("messages", [])
        print(
            f"    [{j}] step={tup.metadata.get('step'):<3} "
            f"source={tup.metadata.get('source'):<10} "
            f"msgs={len(msgs)}"
        )

if parent_history:
    print("\n--- parent head: last message ---")
    last = (parent_history[0].values.get("messages") or [])
    if last:
        last[-1].pretty_print()


discovered 3 (thread_id, checkpoint_ns) location(s):
  thread=ckpt-mode-own_same_thread  ns='__empty__'
  thread=ckpt-mode-own_same_thread  ns='tools:08060305-8228-1835-65f0-274a7a10d87b'
  thread=ckpt-mode-own_same_thread  ns='tools:624b5240-a946-7c04-15d8-438d19b7a28b'

parent history length: 15
  [0] step=-1  source=input      next=('__start__',)  msgs=0
  [1] step=0   source=loop       next=('model',)  msgs=1
  [2] step=1   source=loop       next=('tools', 'tools')  msgs=2
  [3] step=-1  source=input      next=('__start__',)  msgs=0
  [4] step=0   source=loop       next=('model',)  msgs=1
  [5] step=-1  source=input      next=('__start__',)  msgs=0
  [6] step=0   source=loop       next=('model',)  msgs=1
  [7] step=1   source=loop       next=('tools',)  msgs=2
  [8] step=2   source=loop       next=('model',)  msgs=3
  [9] step=1   source=loop       next=('tools',)  msgs=2
  [10] step=2   source=loop       next=('model',)  msgs=3
  [11] step=3   source=loop       next=()  msgs=4
  [

In [8]:
import base64
from typing import Any

import orjson
from redis.asyncio import Redis

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import REDIS_URL

MODE = "own_same_thread"
thread_id = f"ckpt-mode-{MODE}"


def try_decode_blob(b64: str) -> tuple[bool, str | bytes | None]:
    try:
        raw = base64.b64decode(b64)
    except Exception as e:
        return False, f"base64: {e}"
    try:
        orjson.loads(raw)
    except Exception:
        return False, raw
    return True, raw


def walk_doc(node: Any, path: str, out: list[dict]) -> None:
    if isinstance(node, dict):
        if "blob" in node and isinstance(node["blob"], str):
            ok, payload = try_decode_blob(node["blob"])
            if not ok:
                out.append({"path": f"{path}.blob", "type": node.get("type"),
                            "channel": node.get("channel"), "err_or_raw": payload})
        if "__bytes__" in node and isinstance(node["__bytes__"], str):
            ok, payload = try_decode_blob(node["__bytes__"])
            if not ok:
                out.append({"path": f"{path}.__bytes__", "type": "bytes-marker",
                            "channel": None, "err_or_raw": payload})
        for k, v in node.items():
            walk_doc(v, f"{path}.{k}", out)
    elif isinstance(node, list):
        for i, v in enumerate(node):
            walk_doc(v, f"{path}[{i}]", out)


PREFIXES = ["checkpoint", "checkpoint_write", "checkpoint_latest"]
totals: dict[str, dict[str, int]] = {p: {"docs": 0, "json": 0, "bad": 0} for p in PREFIXES}
all_bad: list[dict] = []
non_json: dict[str, list[str]] = {p: [] for p in PREFIXES}

async with Redis.from_url(REDIS_URL, decode_responses=False) as r:
    for prefix in PREFIXES:
        keys = [k async for k in r.scan_iter(match=f"{prefix}:{thread_id}*".encode())]
        totals[prefix]["docs"] = len(keys)
        for k in keys:
            ktype = (await r.type(k)).decode()
            if ktype != "ReJSON-RL":
                non_json[prefix].append(f"{k.decode()}  (type={ktype})")
                continue
            totals[prefix]["json"] += 1
            doc = await r.json().get(k)
            if not isinstance(doc, (dict, list)):
                continue
            findings: list[dict] = []
            walk_doc(doc, "", findings)
            if findings:
                totals[prefix]["bad"] += 1
                for f in findings:
                    f["key"] = k.decode()
                    f["prefix"] = prefix
                all_bad.extend(findings)

print(f"{'prefix':<22} {'docs':>6} {'json':>6} {'bad':>6}")
for p, t in totals.items():
    print(f"{p:<22} {t['docs']:>6} {t['json']:>6} {t['bad']:>6}")

for p, lst in non_json.items():
    if lst:
        print(f"\nnon-JSON keys under {p}:")
        for s in lst[:5]:
            print(f"  {s}")

print(f"\ntotal bad blobs: {len(all_bad)}\n")
for entry in all_bad[:6]:
    raw = entry["err_or_raw"]
    print(f"BAD {entry['prefix']}{entry['path']}  type={entry.get('type')}  channel={entry.get('channel')}")
    print(f"  key: {entry['key']}")
    if isinstance(raw, bytes):
        print(f"  len: {len(raw)}")
        print(f"  head: {raw[:200]!r}")
        print(f"  tail: {raw[-120:]!r}")
    else:
        print(f"  err: {raw}")
    print()


prefix                   docs   json    bad
checkpoint                 15     15      0
checkpoint_write           24     24      7
checkpoint_latest           3      0      0

non-JSON keys under checkpoint_latest:
  checkpoint_latest:ckpt-mode-own_same_thread:__empty__  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:08060305-8228-1835-65f0-274a7a10d87b  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:624b5240-a946-7c04-15d8-438d19b7a28b  (type=string)

total bad blobs: 7

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:tools:08060305-8228-1835-65f0-274a7a10d87b:1f147007-28c0-6c24-bfff-88e820726dbd:9183c2d6-f892-eaf2-4d99-86325da7f616:1
  len: 0
  head: b''
  tail: b''

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:__empty__:1f147007-07fa-6773-bfff-fa7db86a91de:f253aab9-4917-3829-93d6-f3106280c3bb:1
  len: 0
  head: b''
